# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaminari19/FlyRank-Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My answer:** This is primarily a **scoring** task. For every page I want a continuous priority score (how much this page would benefit from a refresh right now), not a yes/no label — pages exist on a spectrum of decline, and how many actually get refreshed this month depends on editorial capacity, not a fixed cutoff. The score naturally supports **ranking** too (sort pages by score, work the queue top-down) — ranking is how the output gets *used*, but the model I'm framing produces a score, not a rank position directly. It isn't classification because a hard needs-refresh/doesn't-need-refresh line would throw away exactly the information (degree of decline) that makes the queue useful. It isn't clustering because I'm not looking for groups of similar pages — I'm ordering *all* pages by one actionable quantity.

In [1]:
import os

REPO_URL = "https://github.com/kaminari19/FlyRank-Starter.git"
REPO_DIR = "FlyRank-Starter"

# Clone only if we don't already have it in this Colab session
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), (
    f"Expected {DATA_PATH} but it's missing — check the repo cloned correctly "
    "and that the CSV wasn't accidentally .gitignored out of your fork."
)

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

# Naming the task type explicitly
ML_TASK_TYPE = "scoring"  # continuous priority score per page; ranking is the consumption pattern
print("ML task type for this lane:", ML_TASK_TYPE)

/content/FlyRank-Starter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.7 MB/s eta 0:00:00
Loaded 30,000 rows x 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**My answer:** There's no ground-truth "this page should have been refreshed" label — refreshing is an editorial decision, not something we observe, so I need a **proxy** built from measured history rather than a true outcome. I'm using `trend_direction`, an observed trend over the recent tracking window, and defining `is_declining_label = (trend_direction == "down")` as a **defined rule** on top of that observed signal — not a hand-labeled outcome. I'll be explicit that this is a proxy, not the real target: a page can be declining for reasons a content refresh won't fix (seasonality, a SERP feature change, a competitor), and a page that looks flat can still be worth refreshing. I'm also keeping this proxy separate from columns like `health_score` /
`recommended_action` / `action_type`, which are themselves *derived* scores — using them as model inputs would leak the answer into the features.

In [2]:
assert "trend_direction" in df_raw.columns, (
    "trend_direction column not found — check docs/data-dictionary.md for the current column name."
)

df_raw["is_declining_label"] = (df_raw["trend_direction"] == "down").astype(int)

print(df_raw["is_declining_label"].value_counts(normalize=True).rename("share_of_pages"))

is_declining_label
1    0.542067
0    0.457933
Name: share_of_pages, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**My answer:** **Precision@50** — of the top 50 pages my score ranks highest, what fraction are actually declining (by the proxy label)? I'm not using overall accuracy or AUC because the real action this feeds is a fixed-capacity editorial queue: the content team can only refresh a handful of pages per sprint, so what matters is whether the *top* of the ranked list is worth their time — not how well the model separates pages nobody will ever look at. "Good" means clearly beating the transparent hand-rule baseline this course's reference pipeline reports (Precision@50 ≈ 0.24 on this same data).

In [3]:
def precision_at_k(y_true, scores, k=50):
    """Fraction of the top-k ranked-by-score items that are actually positive (declining)."""
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    order = np.argsort(-scores)[:k]
    return y_true[order].mean()

# If you've already run `python scripts/run_all.py`, sanity-check against the shipped numbers
import json
results_path = "outputs/model_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        print(json.load(f))
else:
    print("Run `python scripts/run_all.py` to generate outputs/model_results.json "
          "and see the baseline vs. model Precision@50 for yourself.")

Run `python scripts/run_all.py` to generate outputs/model_results.json and see the baseline vs. model Precision@50 for yourself.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**My answer:** One row = one content page belonging to one (anonymized) client, with its recent performance/trend features as of the snapshot date. This lane uses the full content-refresh slice — no further filtering — since the whole point is to score every currently published page.

In [4]:
lane_df = df_raw.copy()  # this lane = the full content-refresh slice; one row = one page

print("Shape:", lane_df.shape)
print("\nColumns:\n", list(lane_df.columns))
lane_df.head()

Shape: (30000, 45)

Columns:
 ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**My answer:** A fixed rule like "flag if traffic dropped more than 20%" can only threshold one signal at a time, but decline shows up as *combinations* of trend, position, CTR, content age, and other signals moving together in ways that interact rather than crossing one clean line — and the right cutoff isn't the same for a competitive commercial page as it is for a long-tail informational one. A single hand-picked threshold also only gives a yes/no flag, not the priority ordering the editorial queue actually needs. This isn't a hunch — the course's own reference pipeline runs this exact comparison on this exact data: the transparent hand-rule baseline scores Precision@50 ≈ 0.24, while a trained model (with a client-holdout split) reaches ≈0.7+, roughly a 3x lift, because it learns which *combinations* of signals actually predict decline instead of picking one rule by hand.

In [5]:
# Quick evidence: do a couple of the "obvious" single-signal deltas move independently of each other?
# (Swap in the real trend/delta column names from docs/data-dictionary.md once you've checked them.)
candidate_signal_cols = [c for c in lane_df.columns if "delta" in c.lower() or "trend" in c.lower()]
print("Trend/delta-style columns found:", candidate_signal_cols)

numeric_candidates = lane_df[candidate_signal_cols].select_dtypes(include="number").columns.tolist()
if len(numeric_candidates) >= 2:
    print(lane_df[numeric_candidates].corr())
else:
    print("Check docs/data-dictionary.md for numeric trend/delta columns, "
          "then compare a couple here to see they don't move in lockstep — "
          "that's the interaction a single if-statement threshold can't capture.")

Trend/delta-style columns found: ['trend_direction', 'trend_pct']
Check docs/data-dictionary.md for numeric trend/delta columns, then compare a couple here to see they don't move in lockstep — that's the interaction a single if-statement threshold can't capture.


## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.